# סרטון יום הולדת 70 — אבא
מרכיב את הסרטון ישירות מGoogle Drive ושומר בחזרה.

**הוראות:**
1. הרץ תא 1 — יחבר את Google Drive
2. הרץ את שאר התאים לפי הסדר
3. הסרטון הסופי יישמר אוטומטית בתיקייה `Claude/יומולדת של אבא/` ב-Drive שלך

In [ ]:
# תא 1 — חיבור Google Drive והתקנת כלים
from google.colab import drive
drive.mount('/content/drive')

!apt-get install -y ffmpeg > /dev/null 2>&1
!pip install Pillow -q
print('✓ Ready')

In [ ]:
# תא 2 — הגדרת נתיבים
import os, glob, shutil

# נתיב ב-Drive שלך — שנה אם שונה
DRIVE_BIRTHDAY = '/content/drive/MyDrive/Claude/יומולדת של אבא'

WORK_DIR    = '/content/birthday_video'
PHOTOS_DIR  = f'{WORK_DIR}/photos'
VIDEOS_DIR  = f'{WORK_DIR}/videos'
MUSIC_DIR   = f'{WORK_DIR}/music'
OUTPUT_DIR  = f'{WORK_DIR}/output'
SEGMENTS_DIR= f'{WORK_DIR}/segments'

for d in [PHOTOS_DIR, VIDEOS_DIR, MUSIC_DIR, OUTPUT_DIR, SEGMENTS_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'Drive path exists: {os.path.exists(DRIVE_BIRTHDAY)}')
print(f'Contents: {os.listdir(DRIVE_BIRTHDAY)[:10] if os.path.exists(DRIVE_BIRTHDAY) else "NOT FOUND"}')

In [ ]:
# תא 3 — העתקת תמונות, וידאו ומוזיקה מ-Drive
import shutil

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG'}
VIDEO_EXTS = {'.mp4', '.MP4', '.MOV', '.mov', '.avi', '.AVI'}
MUSIC_NAMES = {'שלומי שבת', 'אבא של שלומי', 'shlomi', 'avitar', 'אביתר', 'אבא אביתר'}

photos_copied = videos_copied = music_copied = 0

for root, dirs, files in os.walk(DRIVE_BIRTHDAY):
    for fname in files:
        src = os.path.join(root, fname)
        ext = os.path.splitext(fname)[1]
        fname_lower = fname.lower()

        # מוזיקה — קבצי mp4 קטנים (< 5MB) או עם שם מתאים
        is_music = (ext in VIDEO_EXTS and
                    os.path.getsize(src) < 5_000_000 and
                    any(k.lower() in fname_lower for k in ['שלומי', 'אביתר', 'shlomi', 'avitar', 'אבא']))

        if ext in IMAGE_EXTS:
            dst = os.path.join(PHOTOS_DIR, fname)
            if not os.path.exists(dst):
                shutil.copy2(src, dst)
            photos_copied += 1
        elif is_music:
            dst = os.path.join(MUSIC_DIR, fname)
            if not os.path.exists(dst):
                shutil.copy2(src, dst)
            music_copied += 1
        elif ext in VIDEO_EXTS:
            dst = os.path.join(VIDEOS_DIR, fname)
            if not os.path.exists(dst):
                shutil.copy2(src, dst)
            videos_copied += 1

print(f'✓ תמונות: {photos_copied}')
print(f'✓ קטעי וידאו: {videos_copied}')
print(f'✓ מוזיקה: {music_copied}')
print(f'  מוזיקה שנמצאה: {os.listdir(MUSIC_DIR)}')

In [ ]:
# תא 4 — הגדרת שמות קבצי מוזיקה (שנה אם צריך)
music_files = sorted(glob.glob(f'{MUSIC_DIR}/*'))
print('קבצי מוזיקה:', music_files)

# בדרך כלל שלומי שבת קודם, אביתר שני
# שנה את הנתיבים אם הסדר שונה:
if len(music_files) >= 2:
    MUSIC1 = music_files[0]
    MUSIC2 = music_files[1]
elif len(music_files) == 1:
    MUSIC1 = music_files[0]
    MUSIC2 = music_files[0]
else:
    raise Exception('לא נמצאו קבצי מוזיקה! בדוק את תיקיית Drive')

print(f'שיר 1: {os.path.basename(MUSIC1)}')
print(f'שיר 2: {os.path.basename(MUSIC2)}')

In [ ]:
# תא 5 — קוד הבנייה הראשי
import subprocess
from PIL import Image

W, H = 1920, 1080
FPS = 25
PHOTO_DURATION = 5

TITLE_TEXT = 'יום הולדת שמח\nאבא שלנו היקר\nיומולדת 70'
END_TEXT   = 'אבא אוהבים אותך\nהמון המון\nמשפחת פיינגרש\nאתה יצרת'

def run(cmd, desc=''):
    print(f'  → {desc or cmd[:80]}')
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(f'  ERROR: {r.stderr[-400:]}')
        return False
    return True

def find_hebrew_font():
    candidates = [
        '/usr/share/fonts/truetype/noto/NotoSansHebrew-Regular.ttf',
        '/usr/share/fonts/truetype/noto/NotoSans-Regular.ttf',
        '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf',
    ]
    for f in candidates:
        if os.path.exists(f):
            return f
    r = subprocess.run("find /usr/share/fonts -name '*.ttf' | head -1",
                       shell=True, capture_output=True, text=True)
    return r.stdout.strip() or '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf'

def get_duration(path):
    r = subprocess.run(f'ffprobe -v quiet -show_entries format=duration -of csv=p=0 "{path}"',
                       shell=True, capture_output=True, text=True)
    try: return float(r.stdout.strip())
    except: return PHOTO_DURATION

def make_title_slide(text, filename, duration=8, bg_color='1a0a00'):
    out = f'{SEGMENTS_DIR}/{filename}'
    if os.path.exists(out): return out
    font = find_hebrew_font()
    lines = text.split('\n')
    line_h = 100
    start_y = (H - len(lines)*line_h) // 2
    filters = []
    for i, line in enumerate(lines):
        esc = line.replace("'", "\\'").replace(':', '\\:')
        size = 90 if i == 0 else 72
        color = 'white' if i < 2 else '#FFD700'
        y = start_y + i*line_h
        filters.append(
            f"drawtext=fontfile='{font}':text='{esc}':fontcolor={color}:"
            f"fontsize={size}:x=(w-text_w)/2:y={y}:shadowcolor=black:shadowx=3:shadowy=3"
        )
    vf = ','.join(filters) + f',fade=t=in:st=0:d=1,fade=t=out:st={duration-1}:d=1'
    cmd = (f'ffmpeg -y -f lavfi -i color=c={bg_color}:{W}x{H}:rate={FPS}:duration={duration} '
           f'-vf "{vf}" -c:v libx264 -preset fast -crf 20 -pix_fmt yuv420p {out}')
    run(cmd, f'slide: {filename}')
    return out

def make_photo_segment(photo_path, idx, duration=PHOTO_DURATION):
    out = f'{SEGMENTS_DIR}/photo_{idx:03d}.mp4'
    if os.path.exists(out): return out
    BIG_W, BIG_H = 2112, 1188
    total_frames = duration * FPS
    scale = (f'scale={BIG_W}:{BIG_H}:force_original_aspect_ratio=decrease,'
             f'pad={BIG_W}:{BIG_H}:(ow-iw)/2:(oh-ih)/2:black,'
             f'scale=trunc(iw/2)*2:trunc(ih/2)*2')
    if idx % 3 == 0:   z = "'min(zoom+0.0006,1.05)'"
    elif idx % 3 == 1: z = "'if(lte(zoom,1.0),1.04,max(1.0,zoom-0.0006))'"
    else:              z = "'1.02'"
    zoom = (f"zoompan=z={z}:d={total_frames}:"
            f"x='iw/2-(iw/zoom/2)':y='ih/2-(ih/zoom/2)':s={W}x{H}:fps={FPS}")
    fade = f'fade=t=in:st=0:d=0.5,fade=t=out:st={duration-0.5}:d=0.5'
    vf = f'{scale},{zoom},{fade}'
    cmd = (f'ffmpeg -y -loop 1 -i "{photo_path}" -vf "{vf}" '
           f'-t {duration} -r {FPS} -c:v libx264 -preset fast -crf 20 -pix_fmt yuv420p {out}')
    ok = run(cmd, f'photo {idx+1}: {os.path.basename(photo_path)}')
    if not ok and os.path.exists(out) and os.path.getsize(out) == 0:
        os.remove(out)
    return out

def make_video_segment(video_path, idx, max_duration=25):
    out = f'{SEGMENTS_DIR}/video_{idx:03d}.mp4'
    if os.path.exists(out): return out
    dur = get_duration(video_path)
    trim = min(dur, max_duration)
    start = (dur - max_duration) / 4 if dur > max_duration else 0
    vf = (f'scale={W}:{H}:force_original_aspect_ratio=decrease,'
          f'pad={W}:{H}:(ow-iw)/2:(oh-ih)/2:black,'
          f'fade=t=in:st=0:d=0.5,fade=t=out:st={trim-0.5}:d=0.5')
    cmd = (f'ffmpeg -y -ss {start:.2f} -i "{video_path}" -t {trim:.2f} '
           f'-vf "{vf}" -r {FPS} -c:v libx264 -preset fast -crf 20 -pix_fmt yuv420p -an {out}')
    run(cmd, f'video {idx+1}: {os.path.basename(video_path)}')
    return out

def mix_music(total_dur):
    out = f'{WORK_DIR}/mixed_audio.aac'
    if os.path.exists(out): return out
    d1 = get_duration(MUSIC1)
    d2 = get_duration(MUSIC2)
    p1 = min(total_dur * 0.55, d1)
    p2 = min(total_dur - p1, d2)
    fade = 3.0
    cmd = (f'ffmpeg -y -i "{MUSIC1}" -i "{MUSIC2}" -filter_complex '
           f'"[0:a]atrim=0:{p1},asetpts=PTS-STARTPTS,afade=t=in:st=0:d={fade},afade=t=out:st={p1-fade}:d={fade}[a1];'
           f'[1:a]atrim=0:{p2},asetpts=PTS-STARTPTS,afade=t=in:st=0:d={fade},afade=t=out:st={p2-fade}:d={fade}[a2];'
           f'[a1][a2]concat=n=2:v=0:a=1[aout]" '
           f'-map [aout] -c:a aac -b:a 192k {out}')
    run(cmd, f'mixing music ({p1:.0f}s + {p2:.0f}s)')
    return out

print('✓ Functions defined')

In [ ]:
# תא 6 — בנייה
photos = sorted(glob.glob(f'{PHOTOS_DIR}/*'))
videos = sorted(glob.glob(f'{VIDEOS_DIR}/*.mp4') +
                glob.glob(f'{VIDEOS_DIR}/*.MP4') +
                glob.glob(f'{VIDEOS_DIR}/*.MOV') +
                glob.glob(f'{VIDEOS_DIR}/*.mov'))
print(f'תמונות: {len(photos)}, קטעי וידאו: {len(videos)}')

segments = []

print('\n--- שקף פתיחה ---')
segments.append(make_title_slide(TITLE_TEXT, 'title.mp4', duration=8, bg_color='1a0a00'))

print('\n--- תמונות וסרטונים ---')
vi = 0
for i, photo in enumerate(photos):
    segments.append(make_photo_segment(photo, i))
    if (i+1) % 6 == 0 and vi < len(videos):
        segments.append(make_video_segment(videos[vi], vi))
        vi += 1
while vi < len(videos):
    segments.append(make_video_segment(videos[vi], vi))
    vi += 1

print('\n--- שקף סיום ---')
segments.append(make_title_slide(END_TEXT, 'end.mp4', duration=10, bg_color='0a0a1a'))

valid = [s for s in segments if os.path.exists(s) and os.path.getsize(s) > 0]
total_dur = sum(get_duration(s) for s in valid)
print(f'\n{len(valid)} סגמנטים, משך כולל: {total_dur:.0f}ש ({total_dur/60:.1f} דקות)')

In [ ]:
# תא 7 — שרשור ומוזיקה
list_file = f'{WORK_DIR}/concat_list.txt'
with open(list_file, 'w') as f:
    for s in valid:
        f.write(f"file '{s}'\n")

silent = f'{WORK_DIR}/silent_video.mp4'
run(f'ffmpeg -y -f concat -safe 0 -i "{list_file}" '
    f'-c:v libx264 -preset fast -crf 20 -pix_fmt yuv420p {silent}',
    'שרשור סגמנטים')

music = mix_music(total_dur)

FINAL = f'{OUTPUT_DIR}/birthday_final.mp4'
run(f'ffmpeg -y -i "{silent}" -i "{music}" '
    f'-c:v copy -c:a aac -b:a 192k -shortest {FINAL}',
    'הוספת מוזיקה')

size = os.path.getsize(FINAL)/1024/1024
dur  = get_duration(FINAL)
print(f'\n✓ הסרטון מוכן! {size:.0f}MB, {dur/60:.1f} דקות')

In [ ]:
# תא 8 — שמירה ל-Drive
import shutil

DRIVE_OUT = f'{DRIVE_BIRTHDAY}/birthday_final.mp4'
shutil.copy2(FINAL, DRIVE_OUT)
print(f'✓ נשמר ב-Drive: {DRIVE_OUT}')
print(f'  גודל: {os.path.getsize(DRIVE_OUT)/1024/1024:.0f}MB')